# Munti — seed replicate of the ablation

The ablation gap at seed 1337 was +0.039 nats at the final step, but it swings
between +0.037 and +0.063 across the last five evals of the *same* pair of runs.
That is eval noise of a size comparable to the effect, so a single final-step
number is not a defensible measurement.

This retrains both arms at seed 1338 — identical in every other respect,
including the token stream, which is mounted from the original run. Two seeds
won't give a confidence interval, but they will show whether the effect's
*sign and rough size* reproduce, which is the claim the case study actually
makes.

Both arms run in one session: ~45 min each, comfortably inside the 9h limit.

In [ ]:
import os, shutil, subprocess, sys, glob, json
from pathlib import Path

WORK = Path("/kaggle/working/munti-repo")
if not WORK.exists():
    # Prefer the dataset. Mounted kernel outputs each contain a full copy of the
    # repo too, so a bare "first pyproject.toml anywhere" grabs whichever stale
    # snapshot the glob happens to hit first.
    roots = [Path(p).parent for p in glob.glob("/kaggle/input/munti-source/**/pyproject.toml", recursive=True)]
    roots += [Path(p).parent for p in glob.glob("/kaggle/input/*/**/pyproject.toml", recursive=True)]
    assert roots, "repo not found"
    shutil.copytree(roots[0], WORK)
os.chdir(WORK); sys.path.insert(0, str(WORK))

for need in ("configs/munti-12m-seed2.yaml", "configs/ablation-nopos-seed2.yaml"):
    assert (WORK / need).exists(), f"{need} missing — re-upload the dataset with -r zip"

# Same token stream as seed 1337, so the seed is genuinely the only difference.
(WORK / "data").mkdir(exist_ok=True)
for name in ("train.bin", "val.bin", "tokenizer.json"):
    src = next((p for p in glob.glob(f"/kaggle/input/*/**/data/{name}", recursive=True)), None)
    assert src, f"{name} not found — attach the munti-train kernel output"
    shutil.copy(src, WORK / "data" / name)

import torch
cap = torch.cuda.get_device_capability()
print("gpu:", torch.cuda.get_device_name(0), f"sm_{cap[0]}{cap[1]}")
assert cap >= (7, 0), "needs sm_70+; push with --accelerator NvidiaTeslaT4"

print(subprocess.run([sys.executable, "test_munti.py"], capture_output=True, text=True).stdout[-260:])

In [ ]:
from munti.train import train

train("configs/munti-12m-seed2.yaml", resume=False)      # baseline, seed 1338
train("configs/ablation-nopos-seed2.yaml", resume=False)  # ablation, seed 1338

In [ ]:
# Report the gap the honest way: averaged over the last N evals, with the range,
# for both seeds. A single final-step number hides the eval noise entirely.
import csv, statistics

def tail_vals(path, n=5):
    rows = list(csv.DictReader(open(path)))
    return [float(r["val_loss"]) for r in rows[-n:]]

runs = {
    "seed1338": ("out-seed2/loss.csv", "out-nopos-seed2/loss.csv"),
}
# Seed 1337's curves come from the mounted earlier kernels.
base37 = next((p for p in glob.glob("/kaggle/input/*/**/out/loss.csv", recursive=True)), None)
abl37 = next((p for p in glob.glob("/kaggle/input/*/**/out-nopos/loss.csv", recursive=True)), None)
if base37 and abl37:
    runs["seed1337"] = (base37, abl37)

summary = {}
for seed, (b, a) in sorted(runs.items()):
    bv, av = tail_vals(b), tail_vals(a)
    gaps = [x - y for x, y in zip(av, bv)]
    summary[seed] = {
        "baseline_mean": round(statistics.mean(bv), 4),
        "nopos_mean": round(statistics.mean(av), 4),
        "gap_mean": round(statistics.mean(gaps), 4),
        "gap_min": round(min(gaps), 4),
        "gap_max": round(max(gaps), 4),
    }
    print(seed, summary[seed])

Path("out-seed2/ablation_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

In [ ]:
for d in ("out-seed2", "out-nopos-seed2"):
    shutil.make_archive(f"/kaggle/working/{d}", "zip", d)
    print(d, sorted(os.listdir(d)))